# Анализ журнала сделок

Ноутбук только читает `data/trade_journal.csv` и не обращается к Bitunix. Запускайте ячейки сверху вниз.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-darkgrid")

## 1. Загрузка CSV

Путь определяется автоматически при запуске Jupyter из корня проекта или папки `notebooks`.

In [ ]:
cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
journal_path = repo_root / "data" / "trade_journal.csv"

if not journal_path.exists():
    raise FileNotFoundError(f"Журнал пока не найден: {journal_path}")

journal_path

In [ ]:
expected_columns = [
    "event_type", "status", "symbol", "side", "order_type",
    "entry_price", "quantity", "leverage", "stop_loss",
    "take_profit", "client_id", "order_id", "position_id",
    "pnl", "fee", "funding", "net_pnl", "remaining_quantity",
    "source_event_id", "event_id", "timestamp",
]
numeric_columns = [
    "entry_price", "quantity", "leverage", "stop_loss",
    "take_profit", "pnl", "fee", "funding", "net_pnl",
    "remaining_quantity",
]

journal = pd.read_csv(journal_path, dtype=str, keep_default_na=False)
for column in expected_columns:
    if column not in journal.columns:
        journal[column] = ""
for column in numeric_columns:
    journal[column] = pd.to_numeric(journal[column], errors="coerce")
journal["timestamp"] = pd.to_datetime(
    journal["timestamp"], errors="coerce", utc=True
)
journal = journal.sort_values("timestamp", na_position="last").reset_index(drop=True)

print(f"Файл: {journal_path}")
print(f"Строк: {len(journal):,}")
print(f"Период: {journal['timestamp'].min()} — {journal['timestamp'].max()}")
display(journal.tail(10))

## 2. Основные показатели

Расчёты строятся по `TRADE_SUMMARY`, поэтому закрытая позиция учитывается один раз.

In [ ]:
trades = journal.loc[journal["event_type"].eq("TRADE_SUMMARY")].copy()
trades = trades.drop_duplicates("source_event_id", keep="last")

closed_count = len(trades)
wins = int(trades["net_pnl"].gt(0).sum())
losses = int(trades["net_pnl"].lt(0).sum())
breakeven = int(trades["net_pnl"].eq(0).sum())
win_rate = wins / closed_count * 100 if closed_count else 0

metrics = pd.Series({
    "Закрытых сделок": closed_count,
    "Прибыльных": wins,
    "Убыточных": losses,
    "Без результата": breakeven,
    "Win rate, %": round(win_rate, 2),
    "Realized PnL": trades["pnl"].sum(min_count=1),
    "Комиссии": trades["fee"].abs().sum(min_count=1),
    "Funding": trades["funding"].sum(min_count=1),
    "Чистый PnL": trades["net_pnl"].sum(min_count=1),
    "Средний PnL": trades["net_pnl"].mean(),
    "Лучший результат": trades["net_pnl"].max(),
    "Худший результат": trades["net_pnl"].min(),
})
display(metrics.to_frame("Значение"))

## 3. Последние завершённые сделки

In [ ]:
trade_columns = [
    "timestamp", "symbol", "side", "quantity", "entry_price",
    "pnl", "fee", "funding", "net_pnl", "position_id",
]
display(trades[trade_columns].sort_values("timestamp", ascending=False).head(20))

## 4. Накопительный чистый PnL

In [ ]:
equity = trades.dropna(subset=["timestamp", "net_pnl"]).sort_values("timestamp").copy()
equity["cumulative_net_pnl"] = equity["net_pnl"].cumsum()

if equity.empty:
    print("Пока нет завершённых сделок с рассчитанным net_pnl.")
else:
    ax = equity.plot(
        x="timestamp", y="cumulative_net_pnl", figsize=(12, 4),
        title="Накопительный чистый PnL", legend=False,
    )
    ax.set_xlabel("Время")
    ax.set_ylabel("PnL")
    plt.show()

## 5. Результаты по торговым парам

In [ ]:
if trades.empty:
    print("Пока нет завершённых сделок.")
else:
    by_symbol = trades.groupby("symbol", dropna=False).agg(
        trades=("position_id", "count"),
        wins=("net_pnl", lambda values: values.gt(0).sum()),
        net_pnl=("net_pnl", "sum"),
        average_pnl=("net_pnl", "mean"),
        fees=("fee", lambda values: values.abs().sum()),
        funding=("funding", "sum"),
    )
    by_symbol["win_rate_pct"] = by_symbol["wins"] / by_symbol["trades"] * 100
    display(by_symbol.sort_values("net_pnl", ascending=False))

## 6. Результаты по дням

In [ ]:
daily_source = trades.dropna(subset=["timestamp"]).copy()
daily_source["day"] = daily_source["timestamp"].dt.tz_convert("Europe/Moscow").dt.date
daily = daily_source.groupby("day").agg(
    trades=("position_id", "count"),
    net_pnl=("net_pnl", "sum"),
    fees=("fee", lambda values: values.abs().sum()),
    funding=("funding", "sum"),
)
display(daily.sort_index(ascending=False).head(30))

## 7. Исполнения ENTRY, TP и SL

In [ ]:
execution_mask = journal["event_type"].eq("ENTRY") | journal["event_type"].eq("SL") | journal["event_type"].str.fullmatch(r"TP[1-5]", na=False)
executions = journal.loc[execution_mask, [
    "timestamp", "event_type", "symbol", "side", "entry_price",
    "take_profit", "stop_loss", "quantity", "remaining_quantity",
    "pnl", "fee", "funding", "order_id", "position_id",
]]
display(executions.sort_values("timestamp", ascending=False).head(50))

## 8. Проверка качества данных

Показывает пропущенные ID, дубли событий и незаполненные итоговые показатели.

In [ ]:
quality = pd.Series({
    "Дубли source_event_id": journal.loc[journal["source_event_id"].ne(""), "source_event_id"].duplicated().sum(),
    "TP/SL без order_id": executions.loc[executions["event_type"].ne("ENTRY"), "order_id"].eq("").sum(),
    "Итоги без position_id": trades["position_id"].eq("").sum(),
    "Итоги без net_pnl": trades["net_pnl"].isna().sum(),
    "Некорректные timestamp": journal["timestamp"].isna().sum(),
})
display(quality.to_frame("Количество"))

## 9. Экспорт завершённых сделок

Раскомментируйте строку записи, если нужен отдельный компактный CSV.

In [ ]:
export_path = repo_root / "data" / "trade_summary_export.csv"
# trades[trade_columns].to_csv(export_path, index=False)
export_path